# Análisis de Q · G/F

G/F · química 16_2, regla de crecimiento del EKF 16 y protocolo sin umbral del EKF 21. El reparto de YAN del EKF 17 está desactivado. Solo se asimila CO₂. Se requieren dos ventanas consecutivas válidas de 30 minutos. Se conserva el hueco observado de LAB022–024.

Los ajustes de Q usan LAB016–021. Se excluye una fermentación completa en cada ronda. Si el lote excluido es LAB019–021, se ajustan kLa y ganancia con los otros dos de ese grupo. Para la comprobación tardía se ajusta Q con los seis lotes y se usa la pareja de gas archivada. LAB022 es informativo. LAB023–024 son la comprobación tardía fuera del ajuste.

## Ecuaciones y propagación

En G/F, las amplitudes multiplican las sensibilidades discretas a μ₀, βG₀ y βF₀. En S corresponden a crecimiento, fermentación y CO₂ adicional. Las dos columnas idénticas de fermentación se combinan conservando su varianza total. El cálculo requiere un kernel con Numba y las dependencias de lectura de los notebooks de referencia.

In [ ]:
IS_S = False
from numba import njit

@njit
def rates(w, thermal, s0, gas, p, is_s):
    x, n = max(w[0], 0.), max(w[1], 0.)
    g = max(w[2], 0.)
    f = 0. if is_s else max(w[3], 0.)
    sugar = g + f
    e, cd = max(w[-3], 0.), max(w[-2], 0.)
    t, tk, am, ab, ak, maintenance = thermal
    mu = p[0]*am*n/(n+p[3]*ak+1e-8) if sugar > 0. else 0.
    inhibition = 1+e/(p[7]*(ak+1e-8))
    bg = p[1]*ab*g/(g+p[4]*ak+1e-8)/inhibition
    bf = 0. if is_s else p[2]*ab*f/(f+p[5]*ak+1e-8)/(1+g/(p[6]*(ak+1e-8)))/inhibition
    death_temp = -.0001*e**3+.0049*e**2-.1279*e+315.89
    kd = p[8]*np.exp(.0415*e+130000*(tk-305.65)/(305.65*8.314*tk)) if tk >= death_temp else 0.
    ramp = min(max((s0-sugar-5.)/5., 0.), 1.)
    a = ramp*ramp*(3-2*ramp)
    production = a*(mu/12.72+(bg+bf)/1.05)*x
    alcohol = e/7.8924
    saturation = 3.295*np.exp((7e-5*alcohol-.02905)*t-.0179*alcohol-.00111*sugar)
    absorption = gas[0]*max(saturation-cd, 0.)
    transfer = min(production, absorption)-gas[0]*max(cd-saturation, 0.)
    out = np.empty_like(w)
    out[0] = (mu-kd)*x
    out[1] = -mu*x/p[9]
    mg = maintenance*g/(sugar+1.+(0. if is_s else 1e-8))
    out[2] = -((mu/p[10] if g > 0. else 0.)+bg/p[12]+mg)*x
    if not is_s:
        mf = maintenance*f/(sugar+1.+1e-8)
        out[3] = -((mu/p[11] if f > 0. else 0.)+bf/p[13]+mf)*x
    out[-3] = (bg+bf)*x
    out[-2] = transfer
    out[-1] = gas[1]*(production-transfer)
    return out, a, production < absorption

@njit
def advance(w, first, stop, steps, thermal, s0, gas, p, is_s):
    for i in range(first, stop):
        h = steps[i]
        k1 = rates(w, thermal[i, 0], s0, gas, p, is_s)[0]
        k2 = rates(w+h*k1/2, thermal[i, 1], s0, gas, p, is_s)[0]
        k3 = rates(w+h*k2/2, thermal[i, 1], s0, gas, p, is_s)[0]
        k4 = rates(w+h*k3, thermal[i, 2], s0, gas, p, is_s)[0]
        w = np.maximum(w+h*(k1+2*k2+2*k3+k4)/6, 0.)
    return w

@njit
def single_noise(w, temp, s0, gas, p, scale, amp, dt):
    _, activation, dissolved = rates(w, temp, s0, gas, p, True)
    L = np.zeros((6, 3))
    if w[1] > .001 and w[2] > .1:
        L[0,0], L[1,0], L[2,0] = 1., -1/p[9], -1/p[10]
    L[2,1], L[3,1] = -1., p[12]
    production = activation*np.array([1/12.72, p[12]/1.05, 1.])
    L[4 if dissolved else 5] = production*(1. if dissolved else gas[1])
    if w[1] <= .001 or w[2] <= .1:
        L[:,0] = 0.
    scaled = L*amp/scale.reshape(-1, 1)
    return dt*(scaled@scaled.T)

@njit
def replay(grid, doses, measured, is_window, update, initial, P0, scale, offsets,
           steps, thermal, gas, p, amp, R, is_s, correct):
    z = initial/scale
    P = P0/np.outer(scale,scale)
    n = len(z)
    s0 = initial[2] if is_s else initial[2] + initial[3]
    states = np.empty((len(grid),n))
    diag = np.full((len(grid),6), np.nan)
    exhausted = False
    identity = np.eye(n)
    for i in range(len(grid)):
        if i:
            first, stop = offsets[i-1], offsets[i]
            prior = z.copy()
            z = advance(prior*scale, first, stop, steps, thermal, s0, gas, p, is_s)/scale
            if correct:
                J = np.empty((n,n))
                for j in range(n):
                    plus, minus = prior.copy(), prior.copy()
                    plus[j] += 1e-5
                    if prior[j] >= 1e-5:
                        minus[j] -= 1e-5
                    hi = advance(plus*scale,first,stop,steps,thermal,s0,gas,p,is_s)
                    lo = advance(minus*scale,first,stop,steps,thermal,s0,gas,p,is_s)
                    J[:,j] = (hi-lo)/(scale*(2e-5 if prior[j]>=1e-5 else 1e-5))
                if is_s:
                    Q = single_noise(prior*scale, thermal[first,0], s0, gas, p, scale,
                                     amp, grid[i]-grid[i-1])
                else:
                    Gp = np.empty((n,3))
                    for j in range(3):
                        h = p[j]*1e-4
                        plus, minus = p.copy(), p.copy()
                        plus[j], minus[j] = p[j]+h, p[j]-h
                        hi = advance(prior*scale,first,stop,steps,thermal,s0,gas,plus,False)
                        lo = advance(prior*scale,first,stop,steps,thermal,s0,gas,minus,False)
                        Gp[:,j] = (hi-lo)/(2*h*scale)*amp[j]
                    Q = Gp@Gp.T
                P = J@P@J.T+Q
                P = (P+P.T)/2
        z[1] += doses[i]/scale[1]
        if is_window[i]:
            prediction = 2*z[-1]
            innovation = measured[i]-prediction
            variance = 4*P[-1,-1]+R
            clips, delta_yan, delta_sugar = 0, 0., 0.
            if correct and update[i]:
                K = 2*P[:,-1]/variance
                if is_s and exhausted:
                    K[2] *= .5
                raw = z+K*innovation
                clips = np.sum(raw[:-1]<0.)
                positive = np.maximum(raw,0.)
                delta_yan = (positive[1]-z[1])*scale[1]*1000.
                delta_sugar = (positive[2]-z[2])*scale[2]
                if not is_s:
                    delta_sugar += (positive[3]-z[3])*scale[3]
                B = identity.copy()
                B[:,-1] -= 2*K
                P = B@P@B.T+np.outer(K,K)*R
                P = (P+P.T)/2
                z = positive
            diag[i] = np.array([prediction, innovation, variance, clips, delta_yan, delta_sugar])
            z[-1] = 0.
            P[-1,:] = 0.
            P[:,-1] = 0.
        if z[1]*scale[1] <= .001:
            exhausted = True
        if not np.all(np.isfinite(z)) or not np.all(np.isfinite(P)):
            raise ValueError("Propagación no finita")
        states[i] = z*scale
    return states, diag

## Datos y parámetros guardados

In [ ]:
from pathlib import Path
import os
from time import perf_counter

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize_scalar

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p/"src/wine_fermentation").is_dir())
EKF = ROOT/"notebooks/EKF"
ORDER = ("LAB019","LAB020","LAB021","LAB016","LAB017","LAB018","LAB022","LAB023","LAB024")
DEVELOPMENT = ORDER[:6]
GAS_TRAINING = ORDER[:3]
LATE = ORDER[6:]

def reference(n, index):
    path, = EKF.glob(f"{n:02}_*.source.ipynb")
    return nbformat.read(path, 4).cells[index].source

def load_cells(n, indices, changes=None):
    context = {"__name__": f"ekf_{n}", "display": lambda *_: None}
    for i in indices:
        source = reference(n, i)
        source = "\n".join(line for line in source.splitlines() if not line.startswith("%"))
        if changes is not None:
            source = changes(i, source)
        exec(compile(source, f"EKF {n:02} celda {i}", "exec"), context)
    return context

def conversion_density(rho):
    return np.maximum(0., 2.5*(rho-1000.))

sugar_points = {}
def prepare_single():
    def replace_initial(i, source):
        if i == 3:
            assert source.count("initial['G']+initial['F']") == 1
            return source.replace("initial['G']+initial['F']", "0.")
        return source
    ns = load_cells(20, (1, 3, 5), replace_initial)
    data = {}
    for lab, inp in ns["data"].items():
        source = ns["v"].density.rename(columns={"t_h":"time","density_g_l":"density"}) if lab in DEVELOPMENT else (
            ns["prepared"]["offline"].rename(columns={"hours":"time","Densidad_g_L":"density"}))
        rows = source.loc[source.lab.eq(lab), ["sample","time","density"]].dropna()
        first = rows.time.eq(0.) if lab in DEVELOPMENT else rows["sample"].eq("01")
        assert first.sum() == 1 and rows.loc[first,"time"].iloc[0] <= inp["grid"][0]
        density_s0 = conversion_density(float(rows.loc[first,"density"].iloc[0]))
        assert density_s0 > 0.
        initial = inp["initial"].copy()
        initial[2] = density_s0
        obs = rows.loc[~first & rows.time.between(inp["grid"][0],inp["grid"][-1])]
        sugar_points[lab] = (obs.time.to_numpy(), conversion_density(obs.density.to_numpy()))
        inp["initial"] = initial
        grid = inp["grid"]
        frame = inp["sensor"]
        ids = np.searchsorted(grid,frame.index)
        np.testing.assert_array_equal(grid[ids],frame.index)
        measured, valid, windows = np.full(len(grid),np.nan), np.zeros(len(grid),bool), np.zeros(len(grid),bool)
        measured[ids], valid[ids], windows[ids] = frame.measured, frame.valid, True
        doses = np.array([inp["pulses"].get(t,0.) for t in grid])
        data[lab] = dict(inp, y=measured, valid=valid, windows=windows, doses=doses,
                         update=np.zeros(len(grid),bool))
    params = ns["parameters"]
    p = np.array([params[k] if k else 0. for k in (
        "mu0","betaS0",None,"Kn0","Ks0",None,None,"Kie0","Kd0","Yxn","Yxs",None,"Yes",None,"m0")])
    return ns, data, p, np.array([ns["KLA"],ns["GAIN"]]), ns["SCALE_EKF"], np.sqrt(ns["R_CO2"])

def prepare_dual():
    # El requisito de entorno de la referencia se sustituye porque esta ejecución usa Numba.
    def allow_numba_env(i, source):
        if i == 2:
            source = "\n".join(line for line in source.splitlines()
                               if not line.startswith(("prefix =","assert prefix.","print(f\"Kernel")))
        return source
    ns = load_cells(18, (2,4,5,7,8), allow_numba_env)
    data = {}
    for lab in ORDER:
        inp = ns["data"][lab]["Sin rellenar"] if lab in LATE else ns["DATA"][lab]
        grid = inp["grid"]
        windows = (grid > 0.) & np.isclose(grid*2, np.round(grid*2), atol=1e-8, rtol=0.)
        data[lab] = dict(inp, windows=windows, update=np.zeros(len(grid),bool))
    keys = "mu0 betaG0 betaF0 Kn0 Kg0 Kf0 Kig0 Kie0 Kd0 Yxn Yxg Yxf Yeg Yef m0".split()
    p = np.array([ns["theta"][k] for k in keys])
    return ns, data, p, np.array([ns["KLA"],ns["GAIN"]]), ns["SCALE"], np.sqrt(ns["R"])

ns, DATA, parameters, gas_ref, scale, Rsd = prepare_single() if IS_S else prepare_dual()
amplitudes = np.array([.003,np.hypot(.08,.08),.01]) if IS_S else ns["sigma_theta"]
names = ("Crecimiento","Fermentación de S","CO₂ adicional") if IS_S else ("μ₀","βG₀","βF₀")
display(pd.Series(ns["parameters"] if IS_S else ns["theta"], name="Química guardada"))
display(pd.Series(amplitudes, index=names, name="Amplitudes de referencia"))

def thermal_constants(t):
    tk = t+273.15
    R = 8.314
    return (t, tk, np.exp(59453*(tk-300)/(300*R*tk)),
            np.exp(11000*(tk-296.15)/(296.15*R*tk)),
            np.exp(46055*(tk-293.15)/(293.15*R*tk)),
            parameters[14]*np.exp(37681*(tk-293.3)/(293.3*R*tk)))

def pack(d):
    # Mismas etapas RK4 y temperatura causal que la referencia de cada modelo.
    if IS_S:
        offsets = ns["thermal_profiles"][d["batch"].batch][0]
        def intervals(start, stop):
            # La referencia divide en los cambios de la señal causal.
            left = np.searchsorted(offsets,start+1e-10,side="right")
            right = np.searchsorted(offsets,stop-1e-10,side="left")
            edges = np.r_[start,offsets[left:right],stop]
            temps = ns["causal_temperature"](d["batch"],edges[:-1]+1e-10)
            for dt,temp in zip(np.diff(edges),temps):
                count = max(1,int(np.ceil(dt*60-1e-7)))
                for _ in range(count):
                    yield dt/count,(temp,temp,temp)
    else:
        def intervals(start, stop):
            count = max(1,int(np.ceil((stop-start)*60-1e-7)))
            h = (stop-start)/count
            for i in range(count):
                t = start+i*h
                idx = np.searchsorted(d["available"],np.array([t,t+h/2,t+h])+1e-12,side="right")-1
                assert np.min(idx) >= 0
                yield h,d["temperature"][idx]
    steps, thermal, offsets_i = [], [], [0]
    for start, stop in zip(d["grid"][:-1],d["grid"][1:]):
        for duration, temps in intervals(start,stop):
            steps.append(duration)
            thermal.append([thermal_constants(t) for t in temps])
        offsets_i.append(len(steps))
    np.testing.assert_allclose(np.cumsum(np.r_[0.,steps])[offsets_i],d["grid"]-d["grid"][0],atol=1e-8)
    window_ids = np.flatnonzero(d["windows"])
    valid = d["valid"][window_ids]
    d["update"][window_ids] = valid & np.r_[False,valid[:-1]]
    assert np.isfinite(d["y"][d["update"]]).all() and not np.any(d["update"] & ~d["valid"])
    return (d["grid"],d["doses"],d["y"],d["windows"],d["update"],d["initial"],d["P0"],scale,
            np.array(offsets_i),np.array(steps),np.array(thermal))

PACKED = {lab: pack(DATA[lab]) for lab in ORDER}

def run(lab, gas, amp=amplitudes, sd=Rsd, correct=True):
    return replay(*PACKED[lab], np.asarray(gas), parameters, np.asarray(amp), sd**2, IS_S, correct)

print(f"Preparación lista · R de referencia = {Rsd**2:.6g}",flush=True)

## Validación por lote y sensibilidad a R

Se evalúan los 27 cruces de factores 0,5, 1 y 2 de las amplitudes actuales. El criterio es la puntuación gaussiana de la innovación previa a cada corrección, con el mismo peso por lote. Se usa como puntuación comparativa en presencia de no linealidad y recortes. El NIS y la autocorrelación son diagnósticos complementarios.

R se mantiene fijo en cada escenario. Los factores de desviación de medición 0,75, 1 y 1,25 prueban la sensibilidad alrededor del R operativo. Los rangos corresponden a puntos a ≤0,02 nats/ventana del mínimo de la cuadrícula. Los extremos de la cuadrícula limitan las conclusiones.

In [ ]:
from itertools import product
from functools import lru_cache

# Comprobación de paridad con los filtros de referencia usando los datos de LAB024.
lab_check = "LAB024"
check_states, check_diag = run(lab_check,gas_ref)
check_input = DATA[lab_check]
if IS_S:
    replay_definition = reference(22, 5).split("\nresults = {}", 1)[0]
    path, = EKF.glob("20_*.source.ipynb")
    ns.update(EKF=EKF,nbformat=nbformat)
    exec(compile(replay_definition, path.name, "exec"), ns)
    window_input = dict(ns["data"][lab_check])
    ids = np.searchsorted(check_input["grid"], window_input["sensor"].index)
    window_input["sensor"] = window_input["sensor"].assign(update=check_input["update"][ids])
    ref_states, ref_gas, _ = ns["replay_half"](window_input, ns["models"]["Q ajustado"])
    target_states = ref_states[:,0,:]*scale
    target_gas = ref_gas.forecast.to_numpy()
    gas_check = check_diag[ids,0]
else:
    (target_states, _, ref_gas), _, _, _ = ns["replay"](check_input, partition_yan=False)
    target_gas = ref_gas[check_input["windows"],0]
    gas_check = check_diag[check_input["windows"],0]
np.testing.assert_allclose(check_states, target_states, atol=4e-4, rtol=8e-5)
np.testing.assert_allclose(gas_check, target_gas, atol=1e-5, rtol=2e-4)
print("Paridad con referencia LAB024 · error máximo en estados y CO₂",
      np.max(np.abs(check_states-target_states)), np.max(np.abs(gas_check-target_gas)), flush=True)

def fit_gas(labs):
    # La ganancia no realimenta los balances a lazo abierto y se ajusta analíticamente.
    observed = [DATA[lab]["y"][DATA[lab]["valid"]] for lab in labs]
    def loss(log_kla, pair=False):
        k = np.exp(log_kla)
        flows = [run(lab, (k,1.), correct=False)[1][DATA[lab]["valid"],0] for lab in labs]
        numerator = sum(np.mean(x*y) for x,y in zip(flows,observed))
        denominator = sum(np.mean(x*x) for x in flows)
        gain = np.clip(numerator/denominator,.05,3.)
        mse = np.mean([np.mean((gain*x-y)**2) for x,y in zip(flows,observed)])
        return (float(k),float(gain)) if pair else mse
    grid = np.linspace(np.log(.001),np.log(20.),30)
    i = int(np.argmin([loss(x) for x in grid]))
    left, right = grid[max(i-1,0)],grid[min(i+1,len(grid)-1)]
    result = minimize_scalar(loss,bounds=(left,right),method="bounded",
                             options={"xatol":1e-5})
    assert result.success
    return loss(min((grid[i],result.x),key=loss),pair=True)

archived = tuple(gas_ref)
gas_by_fold = {lab: fit_gas(tuple(x for x in GAS_TRAINING if x != lab))
               if lab in GAS_TRAINING else archived for lab in DEVELOPMENT}
display(pd.DataFrame([{"Lote excluido":lab, "Lotes de gas":", ".join(x for x in GAS_TRAINING if x != lab),
                       "kLa [1/h]":gas_by_fold[lab][0], "Ganancia":gas_by_fold[lab][1]}
                      for lab in DEVELOPMENT]).set_index("Lote excluido"))
print("Gas de las rondas listo",flush=True)

MULTIPLIERS = tuple(product((.5,1.,2.),repeat=3))
BASELINE = (1.,1.,1.)
SD_FACTORS = (.75,1.,1.25)
DELTA_SCORE = .02  # Sensibilidad práctica en nats/ventana. No es un intervalo de confianza.
def quasi_score(lab, diag):
    z = diag[DATA[lab]["update"]]
    return .5*np.mean(np.log(z[:,2])+z[:,1]**2/z[:,2])

@lru_cache(maxsize=None)
def scored(lab, gas, multipliers, sd_factor):
    diag = run(lab, gas, amplitudes*np.array(multipliers), Rsd*sd_factor)[1]
    return quasi_score(lab,diag)

def select(training, gas, sd_factor):
    assert set(training) <= set(DEVELOPMENT)
    surface = np.array([np.mean([scored(lab,gas,w,sd_factor) for lab in training])
                        for w in MULTIPLIERS])
    best = int(np.argmin(surface))
    near = np.array(MULTIPLIERS)[surface <= surface[best]+DELTA_SCORE]
    return MULTIPLIERS[best], surface, near

fits, profiles, fold_rows, ranges = {}, [], [], []
for factor in SD_FACTORS:
    for lab in (*DEVELOPMENT, "Todos"):
        if lab == "Todos":
            training, gas = DEVELOPMENT, archived
        else:
            training, gas = tuple(n for n in DEVELOPMENT if n != lab),gas_by_fold[lab]
            assert lab not in training and lab not in LATE
        chosen, surface, near = select(training,gas,factor)
        fits[lab,factor] = (chosen,gas)
        row = {"σR ×":factor,"Lote excluido":lab,
               "Lotes de ajuste":", ".join(training)}
        for j,name in enumerate(names):
            row[name] = chosen[j]*amplitudes[j]
            ranges.append({"σR ×":factor,"Lote excluido":lab,"Amplitud":name,
                           "Mínimo cercano":near[:,j].min()*amplitudes[j],
                           "Máximo cercano":near[:,j].max()*amplitudes[j]})
            if lab == "Todos":
                for m in (.5,1.,2.):
                    match = np.array(MULTIPLIERS)[:,j] == m
                    profiles.append({"σR ×":factor,"Amplitud":name,"Factor":m,
                                     "Δ puntuación":np.min(surface[match])-np.min(surface)})
        if lab != "Todos":
            row["Puntuación ref. excluido"] = scored(lab,gas,BASELINE,factor)
            row["Puntuación ajust. excluido"] = scored(lab,gas,chosen,factor)
        fold_rows.append(row)
    print("Ajuste y validación cruzada terminados · σR ×",factor,flush=True)
fold_table = pd.DataFrame(fold_rows)
profile_table = pd.DataFrame(profiles)
range_table = pd.DataFrame(ranges)
display(fold_table.round(6))
display(range_table.loc[range_table["σR ×"].eq(1.)].round(6))
print(fold_table.round(6).to_string(index=False))

## Evaluación fuera del ajuste

En LAB016–021 se usa el Q de la ronda que deja ese lote fuera. En LAB022–024 se usa el candidato ajustado con LAB016–021. Los RMSE químicos se calculan por interpolación en las horas de muestreo, excluyendo las condiciones iniciales.

In [ ]:
# Muestras posteriores al inicio. El azúcar de S siempre viene de densidad.
single = ns if IS_S else None
dual = ns if not IS_S else None
density_points = {lab: {"times":t,"sugar":y} for lab,(t,y) in sugar_points.items()}
code = reference(21,7)
constants = code[:code.index("def normalize_single")]
exec(compile(constants + code[code.index("def lab_samples"):], "Muestras EKF 21","exec"))
model_label = "S" if IS_S else "G/F"
var_idx = {"X":0,"N":1,"S":2,"E":3} if IS_S else {"X":0,"N":1,"G":2,"F":3,"E":4}

def series(states, name):
    if name == "GF":
        return states[:,2]+states[:,3]
    return states[:,var_idx[name]]*(1000. if name=="N" else 1.)

samples = {}
for lab in ORDER:
    grid = DATA[lab]["grid"]
    for variable in VARIABLES[model_label]:
        if variable != "CO2":
            t,y = (np.asarray(a,dtype=float) for a in lab_samples(model_label,lab,variable))
            keep = np.isfinite(t) & np.isfinite(y) & (t>grid[0]) & (t<=grid[-1])
            samples[lab,variable] = t[keep],y[keep]

runs, rows, phases = {}, [], []
for lab in ORDER:
    folds_out = lab if lab in DEVELOPMENT else "Todos"
    tuned, gas = fits[folds_out,1.]
    base_states, base_diag = run(lab,gas)
    new_states, new_diag = run(lab,gas,amplitudes*np.array(tuned))
    free_states, free_diag = run(lab,gas,correct=False)
    runs[lab] = (("Lazo abierto",free_states,free_diag),
                 ("Q de referencia",base_states,base_diag),("Q candidato",new_states,new_diag))
    d = DATA[lab]
    t, valid, update = d["grid"],d["valid"],d["update"]
    [pulse] = t[d["doses"]>0]
    for label, states, diag in runs[lab]:
        filtered = label != "Lazo abierto"
        residual = -diag[:,1]
        active_diag = diag[update]
        normalized = active_diag[:,1]/np.sqrt(active_diag[:,2])
        phase_masks = {"Arranque (≤40 h)":t<=40., "Antes del pulso (>40 h)":(t>40.) & (t<pulse),
                       "Pospulso (<130 h)":(t>=pulse) & (t<130.), "Cola (≥130 h)":t>=130.}
        prior_n = series(states,"N")
        postlow = np.flatnonzero((t>pulse) & (prior_n<=1.))
        sugar = series(states, "S" if IS_S else "GF")
        row = {"LAB":lab,"Modelo":label,"RMSE CO₂":np.sqrt(np.mean(residual[valid]**2)),
               "RMSE CO₂ desde 130 h":np.sqrt(np.mean(residual[valid & (t>=130.)]**2)) if (valid & (t>=130.)).any() else np.nan,
               "NIS medio":np.mean(normalized**2) if filtered else np.nan,
               "AC1 innovación normalizada":np.corrcoef(normalized[:-1],normalized[1:])[0,1] if filtered else np.nan,
               "Máx. corrección YAN [mg N/L]":np.max(active_diag[:,4]) if filtered else np.nan,
               "Máx. |corrección azúcar| [g/L]":np.max(np.abs(active_diag[:,5])) if filtered else np.nan,
               "Recortes (estados)":int(np.sum(active_diag[:,3])) if filtered else 0,
               "YAN final [mg N/L]":prior_n[-1],
               "Máx. YAN tras agotamiento pospulso":prior_n[postlow[0]:].max() if len(postlow) else np.nan,
               "Azúcar final [g/L]":sugar[-1],
               "Primera vez azúcar ≤1 [h]":t[sugar<=1.][0] if (sugar<=1.).any() else np.nan}
        for stand, mask in phase_masks.items():
            used = mask & valid
            if used.any():
                phases.append(dict(LAB=lab,Modelo=label,Etapa=stand,Ventanas=used.sum(),
                                   Sesgo=np.mean(residual[used]),SD=np.std(residual[used])))
        for var in VARIABLES[model_label]:
            if var == "CO2":
                continue
            times,obs = samples[lab,var]
            if len(obs):
                pred = np.interp(times,t,series(states,var))
                row[f"RMSE {var}"] = np.sqrt(np.mean((pred-obs)**2))
        rows.append(row)
summary = pd.DataFrame(rows).set_index(["LAB","Modelo"])
display(summary.round(4))
display(pd.DataFrame(phases).round(4))
cols = ["RMSE CO₂","RMSE CO₂ desde 130 h","RMSE N","YAN final [mg N/L]","Máx. YAN tras agotamiento pospulso"]
print(summary.loc[list(LATE),cols].round(4).to_string())
print("Promedios por lote excluido (LAB016–021)")
print(summary.loc[list(DEVELOPMENT)].groupby("Modelo")[["RMSE CO₂","RMSE X","RMSE N","RMSE S" if IS_S else "RMSE GF"]].mean().round(4).to_string())

## Curvas por lote y sensibilidad de las amplitudes

In [ ]:
styles = {"Lazo abierto":dict(color=".5",ls=":",alpha=.85),
          "Q de referencia":dict(color="#D55E00",ls="--"),
          "Q candidato":dict(color="#0072B2")}
for variable in VARIABLES[model_label]:
    fig,axes = plt.subplots(3,3,figsize=(14,9),layout="constrained")
    title, unit = SYMBOLS[variable]
    for ax,lab in zip(axes.flat,ORDER):
        d = DATA[lab]
        t = d["grid"]
        if variable == "CO2":
            windows = d["windows"]
            ax.plot(t[windows],d["y"][windows],color="black",alpha=.55,label="Medido")
        else:
            tt,yy = samples[lab,variable]
            if len(yy):
                ax.scatter(tt,yy,color="black",s=12,zorder=5,label="Muestras")
        for label,states,diag in runs[lab]:
            if variable == "CO2":
                ax.plot(t[windows],diag[windows,0],label=label,**styles[label])
            else:
                ax.plot(t,series(states,variable),label=label,**styles[label])
        for pulse in t[d["doses"]>0]:
            ax.axvline(pulse,c=".65",ls=":",lw=.7)
        ax.set(title=lab,xlabel="Tiempo [h]",ylabel=f"{title} [{unit}]")
        ax.grid(alpha=.2)
    axes.flat[0].legend(fontsize=8)
    fig.suptitle(title)
    plt.show()

fig,axes = plt.subplots(1,3,figsize=(12,3.3),layout="constrained")
for ax,name in zip(axes,names):
    table = profile_table.loc[profile_table.Amplitud.eq(name)]
    for factor,frame in table.groupby("σR ×"):
        ax.plot(frame.Factor,frame["Δ puntuación"],marker="o",label=f"σR × {factor:g}")
    ax.set(title=name,xscale="log",xticks=(.5,1,2),
           xticklabels=("0,5","1","2"),xlabel="Factor de amplitud",ylabel="Δ puntuación")
    ax.grid(alpha=.2)
axes[0].legend(fontsize=8)
plt.show()

## Cómo leer la cuadrícula de Q

Los factores multiplican **las amplitudes del ruido**. ×0,5 significa media amplitud, ×1 conserva la referencia y ×2 la duplica. Como la varianza es la amplitud al cuadrado, ×2 implica cuatro veces la varianza de esa fuente. La tabla muestra las amplitudes que se usaron para generar cada eje.

Cada panel fija la tercera amplitud. El número dentro de la casilla es cuánto aumenta la puntuación respecto al mejor punto de ese modelo, en **nats por ventana de actualización de CO₂**, con igual peso por lote. **Más cerca de cero es mejor en LAB016–021.** El 0,000 identifica el mínimo del barrido. La línea punteada marca la combinación de referencia ×1, ×1, ×1. El borde naranja marca el candidato elegido en la cuadrícula.

En G/F, el $Q$ de referencia está a **0,019 nats/ventana** del mejor candidato. Con fermentación de glucosa al doble y fructosa en referencia, las tres amplitudes de crecimiento se separan por alrededor de 0,001. La puntuación discrimina poco entre esos tres valores de crecimiento.

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle

factors = (.5, 1., 2.)
axis_names = ("crecimiento", "fermentación de S", "CO₂ adicional") if IS_S else (
    "μ₀ (crecimiento)", "βG₀ (fermentación de glucosa)", "βF₀ (fermentación de fructosa)")
display(pd.DataFrame({f"× {factor:g}": amplitudes*factor for factor in factors},
                     index=axis_names).round(6))

scores = np.empty((3,3,3))
for i,j,k in product(range(3),repeat=3):
    setting = factors[i],factors[j],factors[k]
    scores[i,j,k] = np.mean([scored(lab,archived,setting,1.) for lab in DEVELOPMENT])
best = np.unravel_index(scores.argmin(),scores.shape)
assert tuple(factors[i] for i in best)==fits["Todos",1.][0]
relative = scores-scores.min()
norm = Normalize(0.,relative.max())
fig,axes = plt.subplots(1,3,figsize=(12.8,5.0),layout="constrained")
for k,ax in enumerate(axes):
    panel = relative[:,:,k]
    image = ax.imshow(panel,cmap="YlGnBu_r",norm=norm,origin="lower",aspect="equal")
    for row,column in product(range(3),repeat=2):
        value = panel[row,column]
        rgb = image.cmap(norm(value))[:3]
        luminance = np.dot(rgb,[.2126,.7152,.0722])
        ax.text(column,row,f"{value:.3f}".replace(".",","),ha="center",va="center",
                color="white" if luminance<.5 else "#18212b",fontsize=11)
        if (row,column,k)==best:
            ax.add_patch(Rectangle((column-.48,row-.48),.96,.96,fill=False,lw=2.5,ec="#e57520"))
        if (row,column,k)==(1,1,1):
            ax.add_patch(Rectangle((column-.44,row-.44),.88,.88,fill=False,lw=1.8,ec="#181818",ls="--"))
    ax.set(xticks=(0,1,2),yticks=(0,1,2),xticklabels=("0,5","1","2"),
           yticklabels=("0,5","1","2") if k==0 else ("","",""),
           xlabel=f"Factor de amplitud\n{axis_names[1]}")
    ax.set_title(f"Amplitud de {axis_names[2]} × {factors[k]:g}".replace("0.5","0,5"),fontsize=10)
    if k==0:
        ax.set_ylabel(f"Factor de amplitud\n{axis_names[0]}")
fig.colorbar(image,ax=axes.tolist(),shrink=.8,pad=.018,
             label="Diferencia respecto al mínimo [nats/ventana]")
fig.suptitle("Puntuación de las 27 combinaciones · "+model_label+" · R de referencia")
fig.legend(handles=[Rectangle((0,0),1,1,fill=False,ec="#e57520",lw=2,label="Mejor de la cuadrícula"),
                    Rectangle((0,0),1,1,fill=False,ec="#181818",ls="--",label="Amplitudes de referencia")],
           loc="outside lower center",ncols=2,frameon=False)
plt.show()

## Cómo leer las barras de validación

Cada fila corresponde a un lote que se dejó fuera. Se grafica el valor del candidato de esa ronda menos el valor del $Q$ de referencia. **Izquierda de cero indica mejora. Derecha indica empeoramiento.** La puntuación gaussiana tiene en cuenta el error y su incertidumbre en las ventanas de actualización. El RMSE resume el tamaño del error de pronóstico en las ventanas válidas, en g/L/h.

En G/F, el RMSE aumentó en los seis lotes excluidos con el $R$ de referencia.

In [ ]:
from matplotlib.lines import Line2D

labs = sorted(DEVELOPMENT)
scored_folds = fold_table.loc[fold_table["σR ×"].eq(1.)].set_index("Lote excluido").loc[labs]
delta_score = scored_folds["Puntuación ajust. excluido"]-scored_folds["Puntuación ref. excluido"]
rmse_by_lab = summary["RMSE CO₂"].unstack().loc[labs]
delta_rmse = rmse_by_lab["Q candidato"]-rmse_by_lab["Q de referencia"]

fig,axes = plt.subplots(1,2,figsize=(10.6,4.1),layout="constrained")
for ax,values,title,unit in (
    (axes[0],delta_score,"Puntuación gaussiana","nats/ventana"),
    (axes[1],delta_rmse,"RMSE de CO₂","g/L/h")):
    ax.barh(labs,values,color=np.where(values<0.,"#226c83","#d17029"),height=.62)
    ax.axvline(0,c=".25",lw=.85)
    ax.grid(axis="x",alpha=.22)
    lo,hi = min(0.,values.min()),max(0.,values.max())
    pad = max((hi-lo)*.16,1e-4)
    ax.set(title=title,xlabel=f"Candidato − referencia [{unit}]",xlim=(lo-pad,hi+pad))
    ax.invert_yaxis()
fig.suptitle("Validación por lote excluido · "+model_label+" · R de referencia")
fig.legend(handles=[Line2D([0],[0],color="#226c83",lw=6,label="Mejora"),
                    Line2D([0],[0],color="#d17029",lw=6,label="Empeora")],
           loc="outside lower center",ncols=2,frameon=False)
plt.show()